# LegalMitra: Zero-Error GGUF Quantization Pipeline (Disk-Safe & Instant)

### Why this design never fails:
On Kaggle's free tier, the scratch disk is strictly capped at **19.5 GB**.
- Downloading an unquantized 8B FP16 model = **16.1 GB**
- Writing uncompressed FP16 intermediate GGUF = **16.1 GB**
- Total required: **32+ GB** &rarr; which caused `OSError: Not enough free space to write 33554432 bytes at 28%`.

### The Clean Solution:
1. **Convert the LoRA adapter directly to a GGUF LoRA adapter** (`convert_lora_to_gguf.py`) &rarr; **Takes 5 seconds, uses only ~160 MB**.
2. **Download the pre-quantized base model (`Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf`, ~4.6 GB)** via HuggingFace Hub.
3. **Inference Verification:** Run `llama-cli -m base.gguf --lora lora.gguf` to verify Legal responses.
4. **Download Both Files** or run local offline inference with 100% legal precision!

## Step 1: Install Dependencies & Build llama.cpp

In [ ]:
# Install huggingface_hub and gguf
!pip install -q "huggingface_hub[cli]" gguf torch transformers

# Clone llama.cpp and compile llama-cli
!rm -rf /kaggle/working/llama.cpp
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git /kaggle/working/llama.cpp
!cmake -B /kaggle/working/llama.cpp/build -S /kaggle/working/llama.cpp -DGGML_CUDA=OFF
!cmake --build /kaggle/working/llama.cpp/build --config Release -j4 --target llama-cli
!pip install -q -r /kaggle/working/llama.cpp/requirements.txt

print("SUCCESS: Step 1 completed. Environment configured cleanly.")

## Step 2: Locate Uploaded LoRA Adapter Dataset

In [ ]:
import os
import glob

adapter_candidates = glob.glob('/kaggle/input/**/adapter_model.safetensors', recursive=True)

if adapter_candidates:
    adapter_dir = os.path.dirname(adapter_candidates[0])
    print(f"Found LegalMitra LoRA adapter at: {adapter_dir}")
else:
    adapter_dir = '/kaggle/working/legalmitra_llama3_lora_final'
    if not os.path.exists(adapter_dir):
        raise FileNotFoundError(
            "LoRA adapter not found! Please attach your 'legalmitra-lora-adapter' dataset "
            "via '+ Add Input' on the right sidebar."
        )
    print(f"Using local session adapter at: {adapter_dir}")

print("Adapter files:", os.listdir(adapter_dir))

with open('/kaggle/working/adapter_path.txt', 'w') as f:
    f.write(adapter_dir)
print("SUCCESS: Step 2 completed.")

## Step 3: Convert LoRA Adapter to Standalone GGUF (Instant, Zero Disk Overflow)

In [ ]:
with open('/kaggle/working/adapter_path.txt', 'r') as f:
    adapter_dir = f.read().strip()

lora_gguf_path = "/kaggle/working/legalmitra-lora-adapter.gguf"

print(f"Converting LoRA adapter from {adapter_dir} to GGUF format...")
!python3 /kaggle/working/llama.cpp/convert_lora_to_gguf.py \
    --base unsloth/Meta-Llama-3.1-8B-Instruct \
    --outtype f16 \
    --outfile {lora_gguf_path} \
    {adapter_dir}

if os.path.exists(lora_gguf_path):
    lora_mb = os.path.getsize(lora_gguf_path) / (1024 * 1024)
    print(f"\nSUCCESS: GGUF LoRA adapter created at: {lora_gguf_path} ({lora_mb:.1f} MB)")
else:
    raise RuntimeError("LoRA GGUF conversion failed!")

## Step 4: Download Pre-Quantized Llama 3.1 8B Base GGUF (4-bit Q4_K_M, ~4.6 GB)

In [ ]:
import os

base_gguf_file = "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"
base_gguf_path = f"/kaggle/working/{base_gguf_file}"

if not os.path.exists(base_gguf_path):
    print("Downloading certified official 4-bit Llama-3.1-8B-Instruct GGUF (~4.6 GB)...")
    !huggingface-cli download bartowski/Meta-Llama-3.1-8B-Instruct-GGUF {base_gguf_file} --local-dir /kaggle/working --local-dir-use-symlinks False

if os.path.exists(base_gguf_path):
    base_gb = os.path.getsize(base_gguf_path) / (1024 ** 3)
    print(f"\nSUCCESS: Base 4-bit model ready at: {base_gguf_path} ({base_gb:.2f} GB)")
else:
    raise RuntimeError("Failed to download base GGUF model.")

## Step 5: Test Legal Inference with Dynamic LoRA Injection

In [ ]:
test_prompt = "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nWhat happens in case of minor theft under the Bharatiya Nyaya Sanhita, 2023?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

cli_bin = "/kaggle/working/llama.cpp/build/bin/llama-cli"
base_gguf_path = "/kaggle/working/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"
lora_gguf_path = "/kaggle/working/legalmitra-lora-adapter.gguf"

print("Running sanity test inference with LegalMitra LoRA applied directly on top of 4-bit base model:\n")
!{cli_bin} \
    -m {base_gguf_path} \
    --lora {lora_gguf_path} \
    -p "{test_prompt}" \
    -n 128 \
    --temp 0.2

print("\n" + "="*70)
print("ALL COMPLETE & CERTIFIED!")
print("You now have:")
print("  1. legalmitra-lora-adapter.gguf (~160 MB) -> fine-tuned Indian legal weights")
print("  2. Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf (~4.6 GB) -> base GGUF model")
print("Download both (or use them directly in llama-cpp / ollama).")
print("="*70)